In [15]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import (
    ScalarFormatter,
    FormatStrFormatter,
    MaxNLocator,
    LogLocator,
    FuncFormatter,
    NullFormatter,
)
wbi_file = "/home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/wbi.npy"
demons_file = "/home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/Demons.npy"
bspline_file = "/home/cyf/wbi/Virginia/code_for_paper/Fig2/2.1_simulation/result/BSpline.npy"

wbi_data = np.load(wbi_file, allow_pickle=True).item()
demons_data = np.load(demons_file, allow_pickle=True).item()
bspline_data = np.load(bspline_file, allow_pickle=True).item()

wbi_processed = wbi_data["processed_results"]
demons_processed = demons_data["processed_results"]
bspline_processed = bspline_data["processed_results"]

x_value_map = {
    "art_R": [30,40,50,60,70,80,90,100,110,120],
    "amp": [6.0,7.2,8.4,9.6,10.8,12.0,13.2,14.4,15.6,16.8],
    "noise_level": [1.0,3.0,6.0,9.0,12.0,15.0,18.0,21.0,24.0],
}

In [2]:
def get_curve(proc_dict, group, avg_key, std_key):
    if group not in proc_dict:
        return None, None
    g = proc_dict[group]
    avg = g.get(avg_key, None)
    std = g.get(std_key, None)
    return avg, std


def merge_processed_results(
    wbi_processed,
    demons_processed,
    bspline_processed,
    experiment_groups
):
    merged = {}

    for group in experiment_groups:
        # 以 wbi 的 labels 为主（通常三者是一致的）
        if group in wbi_processed:
            labels = wbi_processed[group]["labels"]
        elif group in demons_processed:
            labels = demons_processed[group]["labels"]
        elif group in bspline_processed:
            labels = bspline_processed[group]["labels"]
        else:
            continue

        merged[group] = {
            "labels": labels,
            "raw": {},
            "warp": {},
            "map": {},
            "demons": {},
            "bspline": {},
        }

        # -------------------------
        # image metrics from wbi
        # -------------------------
        image_metric_names = ["intensity_mse", "intensity_rmse", "intensity_ssim","runtime_sec"]
        for metric in image_metric_names:
            # raw
            avg, std = get_curve(
                wbi_processed, group,
                f"avg_raw_{metric}", f"std_raw_{metric}"
            )
            merged[group]["raw"][metric] = avg
            merged[group]["raw"][metric + "_std"] = std

            # warp
            avg, std = get_curve(
                wbi_processed, group,
                f"avg_warp_{metric}", f"std_warp_{metric}"
            )
            merged[group]["warp"][metric] = avg
            merged[group]["warp"][metric + "_std"] = std

            # map
            avg, std = get_curve(
                wbi_processed, group,
                f"avg_map_{metric}", f"std_map_{metric}"
            )
            merged[group]["map"][metric] = avg
            merged[group]["map"][metric + "_std"] = std

        # -------------------------
        # motion metrics from wbi
        # -------------------------
        motion_metric_names_wbi = [
            "motion_epe_xy_mean",
            "motion_epe_xy_median",
            "motion_rmse_xy",
            "motion_mae_y",
            "motion_mae_x",
            "motion_rmse_y",
            "motion_rmse_x",
            "motion_epe_3d_mean",
            "motion_epe_3d_median",
            "motion_epe_phys_mean",
            "motion_epe_phys_median",
            "motion_rmse_z",
            "motion_mae_z",
        ]

        for metric in motion_metric_names_wbi:
            avg, std = get_curve(
                wbi_processed, group,
                f"avg_warp_{metric}", f"std_warp_{metric}"
            )
            merged[group]["warp"][metric] = avg
            merged[group]["warp"][metric + "_std"] = std

            avg, std = get_curve(
                wbi_processed, group,
                f"avg_map_{metric}", f"std_map_{metric}"
            )
            merged[group]["map"][metric] = avg
            merged[group]["map"][metric + "_std"] = std

            # raw motion 可选，如果你存了就读
            avg, std = get_curve(
                wbi_processed, group,
                f"avg_raw_{metric}", f"std_raw_{metric}"
            )
            merged[group]["raw"][metric] = avg
            merged[group]["raw"][metric + "_std"] = std

        # -------------------------
        # demons metrics
        # -------------------------
        motion_metric_names_peer = [
            "intensity_mse",
            "intensity_rmse",
            "intensity_ssim",
            "runtime_sec",
            "motion_epe_xy_mean",
            "motion_epe_xy_median",
            "motion_rmse_xy",
            "motion_mae_y",
            "motion_mae_x",
            "motion_rmse_y",
            "motion_rmse_x",
            "motion_epe_3d_mean",
            "motion_epe_3d_median",
            "motion_epe_phys_mean",
            "motion_epe_phys_median",
            "motion_rmse_z",
            "motion_mae_z",
        ]

        for metric in motion_metric_names_peer:
            avg, std = get_curve(
                demons_processed, group,
                f"avg_demons_{metric}", f"std_demons_{metric}"
            )
            merged[group]["demons"][metric] = avg
            merged[group]["demons"][metric + "_std"] = std

            avg, std = get_curve(
                bspline_processed, group,
                f"avg_bspline_{metric}", f"std_bspline_{metric}"
            )
            merged[group]["bspline"][metric] = avg
            merged[group]["bspline"][metric + "_std"] = std

    return merged

def plot_multi_method_metric(
    merged_results,
    experiment_groups,
    metric_name,
    ylabel,
    save_dir="figures",
    include_raw=False,
    use_std=True,
    yscale="linear",
    figsize=(5.0, 3.8),
    linewidth=2.0,
    markersize=5.0,
    dpi=300
):
    os.makedirs(save_dir, exist_ok=True)

    plt.rcParams.update({
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
        "axes.linewidth": 1.0,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    method_styles = {
        "warp":    {"marker": "o", "label": "Warp"},
        "map":     {"marker": "s", "label": "Map"},
        "demons":  {"marker": "^", "label": "Demons"},
        "bspline": {"marker": "D", "label": "B-spline"},
        "raw":     {"marker": None, "label": "Raw"},
    }

    methods_to_plot = ["warp", "map", "demons", "bspline"]
    if include_raw:
        methods_to_plot = ["raw"] + methods_to_plot

    for group in experiment_groups:
        if group not in merged_results:
            continue

        data = merged_results[group]
        labels = data["labels"]

        try:
            x = np.array(labels, dtype=float)
            x_tick_labels = [str(v) for v in labels]
        except Exception:
            x = np.arange(len(labels))
            x_tick_labels = labels

        fig, ax = plt.subplots(figsize=figsize)

        for method in methods_to_plot:
            y = data[method].get(metric_name, None)
            y_std = data[method].get(metric_name + "_std", None)

            if y is None:
                continue

            y = np.array(y, dtype=float)
            if y_std is not None:
                y_std = np.array(y_std, dtype=float)

            style = method_styles[method]

            if method == "raw":
                line, = ax.plot(
                    x, y,
                    linestyle="--",
                    linewidth=linewidth,
                    marker=style["marker"],
                    markersize=markersize,
                    label=style["label"]
                )
            else:
                line, = ax.plot(
                    x, y,
                    linewidth=linewidth,
                    marker=style["marker"],
                    markersize=markersize,
                    label=style["label"]
                )

            if use_std and y_std is not None:
                ax.fill_between(
                    x,
                    y - y_std,
                    y + y_std,
                    alpha=0.15
                )

        ax.set_xlabel(group)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{group}: {ylabel}")

        if yscale in ["linear", "log"]:
            ax.set_yscale(yscale)
            formatter = ScalarFormatter(useMathText=False)
            formatter.set_scientific(False)
            formatter.set_useOffset(False)
            ax.yaxis.set_major_formatter(formatter)
            ax.ticklabel_format(axis='y', style='plain')
        ax.set_xticks(x)
        ax.set_xticklabels(x_tick_labels)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.35)
        ax.legend(frameon=False, loc="best")

        fig.tight_layout()

        out_name = f"{group}_{metric_name}"
        if include_raw:
            out_name += "_with_raw"

        pdf_path = os.path.join(save_dir, out_name + ".pdf")
        png_path = os.path.join(save_dir, out_name + ".png")

        fig.savefig(pdf_path)
        fig.savefig(png_path, dpi=dpi)
        plt.close(fig)

        print(f"Saved: {pdf_path}")
        print(f"Saved: {png_path}")


import os
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.ticker import (
    MaxNLocator,
    ScalarFormatter,
    LogLocator,
    NullFormatter,
    FuncFormatter,
)


def plot_multi_method_metric_v2(
    merged_results,
    experiment_groups,
    metric_name,
    ylabel,
    save_dir="figures",
    x_value_map=None,
    reverse_groups=None,
    include_raw=False,
    use_std=True,
    yscale="linear",
    relative_to_raw=False,
    relative_mode="difference",

    # -----------------------------
    # Recommended layout parameters
    # -----------------------------
    axes_size=(3.6, 2.45),
    margins=(1.15, 0.75, 0.35, 0.30),
    # margins = (left, bottom, right, top), unit: inch

    linewidth=2.2,
    markersize=5.5,
    dpi=300,
    save_pdf=True,
    save_png=True,
    legend_outside=False,
    xtick_step=2,
    draw_legend=False,

    # -----------------------------
    # Font parameters
    # -----------------------------
    base_fontsize=11,
    xtick_labelsize=18,
    ytick_labelsize=20,
    legend_fontsize=12,

    # -----------------------------
    # Optional controls
    # -----------------------------
    fixed_ylim=None,
    fixed_yticks=None,
):
    """
    Plot metric curves for multiple methods and experiment groups.

    This version fixes the physical size of the axes area instead of relying on
    tight_layout or bbox_inches='tight'. Therefore, different figures will have
    identical plotting-area size even if y-axis tick labels have different widths.

    Parameters
    ----------
    merged_results : dict
        Result dictionary, e.g. merged_results[group][method][metric_name].

    experiment_groups : list[str]
        Groups to plot, e.g. ["art_R", "amp", "noise_level"].

    metric_name : str
        Name of metric to plot.

    ylabel : str
        Kept for compatibility. This function currently does not display ylabel.

    save_dir : str
        Output directory.

    x_value_map : dict or None
        Optional mapping from group name to x values.

    reverse_groups : list[str] or None
        Groups whose x/y order should be reversed.

    include_raw : bool
        Whether to include raw result.

    use_std : bool
        Whether to draw mean +/- std as shaded area.

    yscale : {"linear", "log"}
        Y-axis scale.

    relative_to_raw : bool
        Whether to plot result relative to raw.

    relative_mode : {"difference", "ratio"}
        Difference mode or ratio mode relative to raw.

    axes_size : tuple[float, float]
        Physical size of the actual plotting axes in inches.
        Example: (3.6, 2.45)

    margins : tuple[float, float, float, float]
        Margins around the axes in inches:
        (left, bottom, right, top)

    fixed_ylim : tuple[float, float] or None
        If provided, force all figures to use the same y-axis range.

    fixed_yticks : list[float] or None
        If provided, force all figures to use the same y ticks.
    """

    os.makedirs(save_dir, exist_ok=True)

    if x_value_map is None:
        x_value_map = {}

    if reverse_groups is None:
        reverse_groups = []

    # ------------------------------------------------------------------
    # Global font and style settings.
    # Font size is in points and does not automatically scale with figsize.
    # ------------------------------------------------------------------
    plt.rcParams.update({
        "font.size": base_fontsize,
        "axes.labelsize": 30,
        "axes.titlesize": 28,
        "xtick.labelsize": xtick_labelsize,
        "ytick.labelsize": ytick_labelsize,
        "legend.fontsize": legend_fontsize,
        "axes.linewidth": 1.1,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    method_styles = {
        "warp":    {"marker": "o", "label": "Warp"},
        "map":     {"marker": "s", "label": "Map"},
        "demons":  {"marker": "^", "label": "Demons"},
        "bspline": {"marker": "D", "label": "B-spline"},
        "raw":     {"marker": None, "label": "Raw"},
    }

    methods_to_plot = ["warp", "map", "demons", "bspline"]

    if include_raw and not relative_to_raw:
        methods_to_plot = ["raw"] + methods_to_plot

    # ------------------------------------------------------------------
    # Compute fixed figure size from axes size and margins.
    # The axes area will have identical physical width/height in every figure.
    # ------------------------------------------------------------------
    axes_width, axes_height = axes_size
    left_margin, bottom_margin, right_margin, top_margin = margins

    fig_width = left_margin + axes_width + right_margin
    fig_height = bottom_margin + axes_height + top_margin

    axes_rect = [
        left_margin / fig_width,
        bottom_margin / fig_height,
        axes_width / fig_width,
        axes_height / fig_height,
    ]

    for group in experiment_groups:
        if group not in merged_results:
            continue

        data = merged_results[group]
        labels = data["labels"]

        if group in x_value_map:
            x = np.array(x_value_map[group], dtype=float)
            x_tick_labels = [str(v) for v in x_value_map[group]]
        else:
            try:
                x = np.array(labels, dtype=float)
                x_tick_labels = [str(v) for v in labels]
            except Exception:
                x = np.arange(len(labels))
                x_tick_labels = [str(v) for v in labels]

        # --------------------------------------------------------------
        # Create fixed-size figure and fixed-size axes.
        # Do not use tight_layout.
        # Do not use bbox_inches='tight' when saving.
        # --------------------------------------------------------------
        fig = plt.figure(figsize=(fig_width, fig_height))
        ax = fig.add_axes(axes_rect)

        raw_y = None
        raw_y_std = None

        if "raw" in data:
            raw_y = data["raw"].get(metric_name, None)
            raw_y_std = data["raw"].get(metric_name + "_std", None)

            if raw_y is not None:
                raw_y = np.array(raw_y, dtype=float)

            if raw_y_std is not None:
                raw_y_std = np.array(raw_y_std, dtype=float)

        for method in methods_to_plot:
            if method not in data:
                continue

            y = data[method].get(metric_name, None)
            y_std = data[method].get(metric_name + "_std", None)

            if y is None:
                continue

            y = np.array(y, dtype=float)

            if y_std is not None:
                y_std = np.array(y_std, dtype=float)

            if relative_to_raw:
                if method == "raw":
                    continue

                if raw_y is None:
                    continue

                if relative_mode == "difference":
                    y = y - raw_y

                    if y_std is not None and raw_y_std is not None:
                        y_std = np.sqrt(y_std ** 2 + raw_y_std ** 2)

                elif relative_mode == "ratio":
                    eps = 1e-12
                    y = (y - raw_y) / (raw_y + eps)

                else:
                    raise ValueError(f"Unknown relative_mode: {relative_mode}")

            x_plot = x.copy()
            y_plot = y.copy()
            y_std_plot = None if y_std is None else y_std.copy()

            if group in reverse_groups:
                x_plot = x_plot[::-1]
                y_plot = y_plot[::-1]

                if y_std_plot is not None:
                    y_std_plot = y_std_plot[::-1]

            style = method_styles.get(method, {"marker": "o", "label": method})

            if method == "raw":
                ax.plot(
                    x_plot,
                    y_plot,
                    linestyle="--",
                    linewidth=linewidth,
                    marker=style["marker"],
                    markersize=markersize,
                    label=style["label"],
                )
            else:
                ax.plot(
                    x_plot,
                    y_plot,
                    linewidth=linewidth,
                    marker=style["marker"],
                    markersize=markersize,
                    label=style["label"],
                )

            if use_std and y_std_plot is not None and relative_mode != "ratio":
                lower = y_plot - y_std_plot
                upper = y_plot + y_std_plot

                if yscale == "log":
                    eps = 1e-12
                    lower = np.maximum(lower, eps)
                    upper = np.maximum(upper, eps)

                ax.fill_between(
                    x_plot,
                    lower,
                    upper,
                    alpha=0.10,
                )

        # ------------------------------------------------------------------
        # X ticks
        # ------------------------------------------------------------------
        x_ticks_final = x[::-1] if group in reverse_groups else x
        x_tick_labels_final = x_tick_labels[::-1] if group in reverse_groups else x_tick_labels

        tick_idx = np.arange(0, len(x_ticks_final), xtick_step)

        ax.set_xticks(x_ticks_final[tick_idx])
        ax.set_xticklabels([x_tick_labels_final[i] for i in tick_idx])

        # Do not show axis labels.
        ax.set_xlabel("")
        ax.set_ylabel("")

        # ------------------------------------------------------------------
        # Y scale and y tick formatter
        # ------------------------------------------------------------------
        if yscale == "linear":
            ax.set_yscale("linear")

            if fixed_ylim is not None:
                ax.set_ylim(fixed_ylim)

            if fixed_yticks is not None:
                ax.set_yticks(fixed_yticks)
            else:
                ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

            formatter = ScalarFormatter(useMathText=False)
            formatter.set_scientific(False)
            formatter.set_useOffset(False)

            ax.yaxis.set_major_formatter(formatter)
            ax.ticklabel_format(axis="y", style="plain", useOffset=False)

        elif yscale == "log":
            ax.set_yscale("log")

            if fixed_ylim is not None:
                ymin, ymax = fixed_ylim
                eps = 1e-12
                ax.set_ylim(max(ymin, eps), ymax)
            else:
                ymin, ymax = ax.get_ylim()
                eps = 1e-12
                ax.set_ylim(max(ymin, eps), ymax)

            if fixed_yticks is not None:
                ax.set_yticks(fixed_yticks)
            else:
                ax.yaxis.set_major_locator(
                    LogLocator(base=10, subs=np.arange(2, 6), numticks=20)
                )

            ax.yaxis.set_minor_formatter(NullFormatter())

            def log_plain_formatter(y, pos):
                if y <= 0:
                    return ""
                return f"{y:g}"

            ax.yaxis.set_major_formatter(FuncFormatter(log_plain_formatter))

        else:
            ax.set_yscale("linear")

        # ------------------------------------------------------------------
        # Tick visibility
        # ------------------------------------------------------------------
        ax.tick_params(axis="x", which="both", labelbottom=True)
        ax.tick_params(axis="y", which="both", labelleft=True)

        ax.yaxis.set_tick_params(labelleft=True)

        for label in ax.get_yticklabels():
            label.set_visible(True)

        # ------------------------------------------------------------------
        # Spines and grid
        # ------------------------------------------------------------------
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)

        ax.grid(
            True,
            axis="y",
            linestyle="--",
            linewidth=0.7,
            alpha=0.25,
        )

        # ------------------------------------------------------------------
        # Axis arrows.
        # Because we do not use bbox_inches='tight', arrowheads outside axes
        # must remain within the figure margins.
        # ------------------------------------------------------------------
        ax.annotate(
            "",
            xy=(1.02, 0.0),
            xytext=(0.0, 0.0),
            xycoords="axes fraction",
            arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"),
            annotation_clip=False,
        )

        ax.annotate(
            "",
            xy=(0.0, 1.02),
            xytext=(0.0, 0.0),
            xycoords="axes fraction",
            arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"),
            annotation_clip=False,
        )

        # ------------------------------------------------------------------
        # Legend.
        # If you want legends, it is better to save a separate legend figure.
        # If legend_outside=True, increase right_margin.
        # ------------------------------------------------------------------
        if draw_legend:
            if legend_outside:
                ax.legend(
                    frameon=False,
                    loc="center left",
                    bbox_to_anchor=(1.05, 0.5),
                    borderaxespad=0.0,
                )
            else:
                ax.legend(frameon=False, loc="best")

        # ------------------------------------------------------------------
        # Save
        # Important:
        # Do not use bbox_inches='tight'.
        # Otherwise, output PDF size will differ across figures.
        # ------------------------------------------------------------------
        suffix = metric_name

        if relative_to_raw:
            suffix += f"_vs_raw_{relative_mode}"

        if group in reverse_groups:
            suffix += "_reversed"

        if yscale == "log":
            suffix += "_logy"

        suffix += "_nolabel"

        if save_pdf:
            pdf_path = os.path.join(save_dir, f"{group}_{suffix}.pdf")
            fig.savefig(pdf_path)
            print(f"Saved: {pdf_path}")

        if save_png:
            png_path = os.path.join(save_dir, f"{group}_{suffix}.png")
            fig.savefig(png_path, dpi=dpi)
            print(f"Saved: {png_path}")

        plt.close(fig)
experiment_groups = ["art_R", "amp", "noise_level"]

merged_results = merge_processed_results(
    wbi_processed=wbi_processed,
    demons_processed=demons_processed,
    bspline_processed=bspline_processed,
    experiment_groups=experiment_groups
)

In [ ]:
## Intensity MSE
plot_multi_method_metric_v2(
    merged_results=merged_results,
    experiment_groups=experiment_groups,
    metric_name="intensity_mse",
    reverse_groups=["art_R"],
    ylabel="",
    save_dir="figures",
    axes_size=(3.6, 2.45),
    margins=(1.35, 0.75, 0.35, 0.30),
    linewidth=2.2,
    markersize=5.5,
    dpi=300,
    draw_legend=False,
    xtick_step=2,
    yscale="linear",
    include_raw=True
    )


Saved: 2.1_simulation_compare/intensity_MSE/art_R_intensity_mse_reversed_logy_nolabel.pdf
Saved: 2.1_simulation_compare/intensity_MSE/art_R_intensity_mse_reversed_logy_nolabel.png
Saved: 2.1_simulation_compare/intensity_MSE/amp_intensity_mse_logy_nolabel.pdf
Saved: 2.1_simulation_compare/intensity_MSE/amp_intensity_mse_logy_nolabel.png
Saved: 2.1_simulation_compare/intensity_MSE/noise_level_intensity_mse_logy_nolabel.pdf
Saved: 2.1_simulation_compare/intensity_MSE/noise_level_intensity_mse_logy_nolabel.png


In [17]:
## Intensity RMSE
plot_multi_method_metric_v2(
    merged_results=merged_results,
    experiment_groups=experiment_groups,
    metric_name="intensity_rmse",
    reverse_groups=["art_R"],
    ylabel="",
    save_dir="2.1_simulation_compare/intensity_RMSE",
    axes_size=(3.6, 2.45),
    x_value_map=x_value_map,
    margins=(1.35, 0.75, 0.35, 0.30),
    linewidth=2.2,
    markersize=5.5,
    include_raw=True,
    dpi=300,
    draw_legend=False,
    xtick_step=2,
    yscale="log",
)

Saved: 2.1_simulation_compare/intensity_RMSE/art_R_intensity_rmse_reversed_logy_nolabel.pdf
Saved: 2.1_simulation_compare/intensity_RMSE/art_R_intensity_rmse_reversed_logy_nolabel.png
Saved: 2.1_simulation_compare/intensity_RMSE/amp_intensity_rmse_logy_nolabel.pdf
Saved: 2.1_simulation_compare/intensity_RMSE/amp_intensity_rmse_logy_nolabel.png
Saved: 2.1_simulation_compare/intensity_RMSE/noise_level_intensity_rmse_logy_nolabel.pdf
Saved: 2.1_simulation_compare/intensity_RMSE/noise_level_intensity_rmse_logy_nolabel.png


In [ ]:
## Intensity SSIM
plot_multi_method_metric_v2(
    merged_results=merged_results,
    experiment_groups=experiment_groups,
    metric_name="intensity_ssim",
    ylabel="Intensity SSIM",
    reverse_groups=["art_R"],
    x_value_map=x_value_map,
    axes_size=(3.6, 2.45),
    margins=(1.35, 0.75, 0.35, 0.30),
    linewidth=2.2,
    markersize=5.5,
    include_raw=True,
    dpi=300,
    draw_legend=False,
    xtick_step=2,
    yscale="linear",
    save_dir="2.1_simulation_compare/intensity_SSIM",
    use_std=True,
    relative_to_raw = False
)

In [ ]:
## motion EPE
plot_multi_method_metric_v2(
    merged_results=merged_results,
    experiment_groups=experiment_groups,
    metric_name="motion_epe_phys_mean",
    ylabel="Motion EPE Physical Mean",
    save_dir="2.1_simulation_compare/motion_EPE",
    reverse_groups=["art_R"],
    figsize=(6, 3),
    x_value_map=x_value_map,
    include_raw=False,
    use_std=True,
    yscale="linear",
    relative_to_raw = False
)

保存图例
---


In [5]:
import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def save_method_legend(
    save_dir="figures",
    filename="methods_legend",
    include_raw=False,
    save_pdf=True,
    save_png=True,
    dpi=300,
    fontsize=12,
    ncol=1,
    figsize=(2.2, 1.8),
):
    """
    Save a standalone legend figure for methods.

    The colors are assigned using matplotlib's default color cycle,
    so they match plot_multi_method_metric_v2 when the method order is the same.
    """

    os.makedirs(save_dir, exist_ok=True)

    method_styles = {
        "warp":    {"marker": "o", "label": "Warp"},
        "map":     {"marker": "s", "label": "Map"},
        "demons":  {"marker": "^", "label": "Demons"},
        "bspline": {"marker": "D", "label": "B-spline"},
        "raw":     {"marker": None, "label": "Raw"},
    }

    methods = ["warp", "map", "demons", "bspline"]
    if include_raw:
        methods = ["raw"] + methods

    # Use the same default matplotlib color cycle as ax.plot()
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    method_colors = {
        method: colors[i % len(colors)]
        for i, method in enumerate(methods)
    }

    handles = []
    for method in methods:
        style = method_styles[method]

        h = Line2D(
            [0], [0],
            linestyle="--" if method == "raw" else "-",
            linewidth=4.0,
            marker=style["marker"],
            markersize=5.5,
            label=style["label"],
            color=method_colors[method],
            markerfacecolor=method_colors[method],
            markeredgecolor=method_colors[method],
        )
        handles.append(h)

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")

    ax.legend(
        handles=handles,
        loc="center",
        frameon=False,
        fontsize=fontsize,
        ncol=ncol,
        handlelength=2.0,
        columnspacing=1.2,
        handletextpad=0.6,
        labelspacing=1.2
    )

    fig.tight_layout()

    if save_pdf:
        pdf_path = os.path.join(save_dir, f"{filename}.pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved: {pdf_path}")

    if save_png:
        png_path = os.path.join(save_dir, f"{filename}.png")
        fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {png_path}")

    plt.close(fig)


save_method_legend(
    save_dir="2.1_simulation_compare",
    filename="methods_legend",
    include_raw=True,
    save_pdf=True,
    save_png=False,
    fontsize=18,
    ncol=1,
    figsize=(5, 2.0),
)

Saved: 2.1_simulation_compare/methods_legend.pdf


计算时间
---

In [70]:
wbi_raw = wbi_data["raw_results"]
demons_raw = demons_data['raw_results']
bspline_raw = bspline_data['raw_results']
demons_raw = demons_data['raw_results']
warp_time = []
map_time = []
bspline_time = []
demons_time = []
for exp in experiment_groups:
    for params in ['1','2','3','4','5','6','7','8','9','10']:
        if params in wbi_raw[exp] and wbi_raw[exp][params]:
            warp_time.append(wbi_raw[exp][params]['warp_runtime_sec'])
            map_time.append(wbi_raw[exp][params]['map_runtime_sec'])
            bspline_time.append(bspline_raw[exp][params]['bspline_runtime_sec'])
            demons_time.append(demons_raw[exp][params]['demons_runtime_sec'])
warp_time = [item for sublist in warp_time for item in sublist]
map_time = [item for sublist in map_time for item in sublist]
bspline_time = [item for sublist in bspline_time for item in sublist]
demons_time = [item for sublist in demons_time for item in sublist]
runtime_dict = {
    "Map": map_time,
    "Warp": warp_time,
    "Demons": demons_time,
    "B-spline": bspline_time,
}

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


def plot_runtime_bar_horizontal(
    runtime_dict,
    save_dir="runtime_figures",
    filename="runtime_bar_horizontal",
    figsize=(7.2, 4.6),
    xlabel="Runtime (s)",
    dpi=300,
    save_pdf=True,
    save_png=True,
    use_log=False,
    error_type="std",   # "std" or "sem"
    show_values=True,
    value_fontsize=14,  # 柱子上数字字体大小
    value_color="black",  # "black" or "match_bar"
):
    """
    Parameters
    ----------
    runtime_dict : dict
        e.g.
        {
            "Warp": warp_times,
            "Map": map_times,
            "Demons": demons_times,
            "B-spline": bspline_times,
        }

    error_type : str
        "std"  -> error bar = standard deviation
        "sem"  -> error bar = standard error

    value_color : str
        "black"     -> 柱子右侧数字统一黑色
        "match_bar" -> 数字颜色和柱子一致
    """

    os.makedirs(save_dir, exist_ok=True)

    method_names = []
    means = []
    errors = []
    counts = []

    # -------------------------
    # flatten and summarize
    # -------------------------
    for method, values in runtime_dict.items():
        flat = []
        for v in values:
            if isinstance(v, (list, tuple, np.ndarray)):
                flat.extend(np.asarray(v).ravel().tolist())
            else:
                flat.append(v)

        arr = np.asarray(flat, dtype=float)
        arr = arr[np.isfinite(arr)]

        if len(arr) == 0:
            continue

        mean_val = np.mean(arr)
        std_val = np.std(arr)
        sem_val = std_val / np.sqrt(len(arr))

        method_names.append(method)
        means.append(mean_val)
        counts.append(len(arr))

        if error_type == "std":
            errors.append(std_val)
        elif error_type == "sem":
            errors.append(sem_val)
        else:
            raise ValueError("error_type must be 'std' or 'sem'")

    method_names = np.array(method_names)
    means = np.array(means)
    errors = np.array(errors)
    counts = np.array(counts)

    # -------------------------
    # style
    # -------------------------
    plt.rcParams.update({
        "font.size": 16,
        "axes.labelsize": 20,
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "axes.linewidth": 1.1,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 10,
        "ytick.major.size": 8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


    # 你可以自己换这一组颜色
    bar_colors = {
        "Warp": "#f0842d",
        "Map": "#439C3D",
        "Demons": "#cb322c",
        "B-spline": "#8963a8",
    }

    # 给每个柱子分配颜色，如果 method 不在上面，就给默认灰色
    colors = [bar_colors.get(m, "#808080") for m in method_names]

    fig, ax = plt.subplots(figsize=figsize)

    y = np.arange(len(method_names))

    bars = ax.barh(
        y,
        means,
        xerr=None,
        height=0.62,
        capsize=5,
        alpha=0.9,
        color=colors,
        edgecolor="none"
    )

    ax.set_yticks(y)
    ax.set_yticklabels(method_names)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")

    if use_log:
        ax.set_xscale("log")

    # 把第一个方法放在最上面
    ax.invert_yaxis()

    # 网格与边框
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis="x", linestyle="--", linewidth=0.7, alpha=0.3)

    # 在柱子右边标数值
    if show_values:
        offset = 0.5

        for yi, mean_val, err_val, method in zip(y, means, errors, method_names):
            if value_color == "match_bar":
                txt_color = bar_colors.get(method, "black")
            else:
                txt_color = "black"

            ax.text(
                mean_val*1.12,
                yi,
                f"{mean_val:.2f}",
                va="center",
                ha="left",
                fontsize=value_fontsize,
                color=txt_color
            )

    fig.tight_layout()

    # 保存
    if save_pdf:
        pdf_path = os.path.join(save_dir, f"{filename}.pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved: {pdf_path}")

    if save_png:
        png_path = os.path.join(save_dir, f"{filename}.png")
        fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {png_path}")

    plt.close(fig)

plot_runtime_bar_horizontal(
    runtime_dict=runtime_dict,
    save_dir="2.1_simulation_compare/runtime",
    filename="runtime_bar_horizontal",
    figsize=(5, 3.5),
    xlabel="Runtime (s)",
    save_pdf=True,
    save_png=True,
    use_log=True,
    value_fontsize=18,
    error_type="std",   # 或者 "sem"
    show_values=True,
)

Saved: 2.1_simulation_compare/runtime/runtime_bar_horizontal.pdf
Saved: 2.1_simulation_compare/runtime/runtime_bar_horizontal.png


计算motion的loss
---

In [73]:
wbi_raw = wbi_data["raw_results"]
demons_raw = demons_data['raw_results']
bspline_raw = bspline_data['raw_results']
demons_raw = demons_data['raw_results']
warp_motionError = []
map_motionError = []
bspline_motionError = []
demons_motionError = []
for exp in experiment_groups:
    for params in ['1','2','3','4','5','6','7','8','9','10']:
        if params in wbi_raw[exp] and wbi_raw[exp][params]:
            warp_motionError.append(wbi_raw[exp][params]['warp_motion_epe_phys_mean'])
            map_motionError.append(wbi_raw[exp][params]['map_motion_epe_phys_mean'])
            bspline_motionError.append(bspline_raw[exp][params]['bspline_motion_epe_phys_mean'])
            demons_motionError.append(demons_raw[exp][params]['demons_motion_epe_phys_mean'])
warp_motionError = [item for sublist in warp_motionError for item in sublist]
map_motionError = [item for sublist in map_motionError for item in sublist]
bspline_motionError = [item for sublist in bspline_motionError for item in sublist]
demons_motionError = [item for sublist in demons_motionError for item in sublist]
runmotionError_dict = {
    "Map": map_motionError,
    "Warp": warp_motionError,
    "Demons": demons_motionError,
    "B-spline": bspline_motionError,

}

In [78]:
import os
import numpy as np
import matplotlib.pyplot as plt


def plot_motion_error_distribution_horizontal(
    error_dict,
    save_dir="motion_error_figures",
    filename="motion_error_distribution",
    figsize=(7.8, 4.8),
    xlabel="Motion Error",
    dpi=300,
    save_pdf=True,
    save_png=True,
    use_log=False,
    show_violin=True,
    show_box=True,
    show_points=True,
    point_size=10,
    point_alpha=0.25,
):
    """
    Parameters
    ----------
    error_dict : dict
        e.g.
        {
            "Warp": warp_motion_error,
            "Map": map_motion_error,
            "Demons": demons_motion_error,
            "B-spline": bspline_motion_error,
        }

        每个 value 可以是:
        - 一维 list / np.ndarray
        - 或者 list[list]，函数会自动展平
    """

    os.makedirs(save_dir, exist_ok=True)

    # -------------------------
    # flatten and clean
    # -------------------------
    method_names = []
    data_list = []

    for method, values in error_dict.items():
        flat = []
        for v in values:
            if isinstance(v, (list, tuple, np.ndarray)):
                flat.extend(np.asarray(v).ravel().tolist())
            else:
                flat.append(v)

        arr = np.asarray(flat, dtype=float)
        arr = arr[np.isfinite(arr)]

        # log轴时必须去掉 <=0
        if use_log:
            arr = arr[arr > 0]

        if len(arr) == 0:
            continue

        method_names.append(method)
        data_list.append(arr)

    # -------------------------
    # style
    # -------------------------
    plt.rcParams.update({
        "font.size": 16,
        "axes.labelsize": 20,
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "axes.linewidth": 1.1,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 10,
        "ytick.major.size": 8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


    # 和你前面 method 保持一致的颜色会更好
    method_colors = {
        "Warp": "#f0842d",
        "Map": "#439C3D",
        "Demons": "#cb322c",
        "B-spline": "#8963a8",
    }

    colors = [method_colors.get(m, "#808080") for m in method_names]

    fig, ax = plt.subplots(figsize=figsize)

    positions = np.arange(1, len(method_names) + 1)

    # -------------------------
    # violin
    # -------------------------
    if show_violin:
        violin = ax.violinplot(
            data_list,
            positions=positions,
            vert=False,
            showmeans=False,
            showmedians=False,
            showextrema=False,
            widths=0.82
        )

        for body, c in zip(violin["bodies"], colors):
            body.set_facecolor(c)
            body.set_edgecolor("none")
            body.set_alpha(0.5)

    # -------------------------
    # boxplot
    # -------------------------
    if show_box:
        bp = ax.boxplot(
            data_list,
            positions=positions,
            vert=False,
            widths=0.28,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.8),
            boxprops=dict(linewidth=1.2, edgecolor="black"),
            whiskerprops=dict(linewidth=1.0, color="black"),
            capprops=dict(linewidth=1.0, color="black"),
        )

        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.55)

    # -------------------------
    # jittered points
    # -------------------------
    if show_points:
        rng = np.random.default_rng(0)
        for i, (arr, c) in enumerate(zip(data_list, colors), start=1):
            y_jitter = rng.normal(loc=i, scale=0.06, size=len(arr))
            ax.scatter(
                arr,
                y_jitter,
                s=point_size,
                alpha=point_alpha,
                color=c,
                edgecolors="none",
                zorder=2
            )

    # -------------------------
    # axis
    # -------------------------
    ax.set_yticks(positions)
    ax.set_yticklabels(method_names)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")

    if use_log:
        ax.set_xscale("log")

    ax.invert_yaxis()  # 第一个方法放最上面

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis="x", linestyle="--", linewidth=0.7, alpha=0.3)

    fig.tight_layout()

    # -------------------------
    # save
    # -------------------------
    if save_pdf:
        pdf_path = os.path.join(save_dir, f"{filename}.pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved: {pdf_path}")

    if save_png:
        png_path = os.path.join(save_dir, f"{filename}.png")
        fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {png_path}")

    plt.close(fig)




plot_motion_error_distribution_horizontal(
    error_dict=runmotionError_dict,
    save_dir="2.1_simulation_compare/motion_error",
    filename="motion_error_distribution",
    figsize=(5, 3.5),
    xlabel="Motion EPE (pixels)",
    save_pdf=True,
    save_png=True,
    use_log=True,      # 如果跨度特别大可以改 True
    show_violin=True,
    show_box=True,
    show_points=True,
    point_size=14,
    point_alpha=0.4,
)

Saved: 2.1_simulation_compare/motion_error/motion_error_distribution.pdf
Saved: 2.1_simulation_compare/motion_error/motion_error_distribution.png
